In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from scipy import signal
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, Dense, Conv1D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

# ================================================
# ROBUST INFLATION FORECASTING WITH BAXTER-KING FILTERING
# Scientific approach using business cycle frequency analysis
# ================================================

print("🚀 ROBUST INFLATION FORECASTING WITH BAXTER-KING FILTERING")
print("="*75)
print("✅ NO inflation as predictor (avoiding data leakage)")
print("✅ Using Baxter-King bandpass filter for business cycle analysis")  
print("✅ Multiple evaluation metrics (MSE, MAE, R², MAPE)")
print("✅ Scientific methodology for policy applications")
print("✅ Business cycle frequency isolation from Fed indicators")
print("="*75)

# ================================================
# 1. SCIENTIFIC SETUP - LEADING FED INDICATORS
# ================================================

def load_and_document_data(file_path):
    """
    Load inflation forecasting data with Fed leading indicators
    
    Leading Indicators for Baxter-King Filter Analysis:
    - USM2: Money Supply M2 (monetary policy cycles)
    - USPPIYY: Producer Price Index YoY (inflation pressure cycles)  
    - INDPRO: Industrial Production Index (business activity cycles)
    - UNRATE: Unemployment Rate (labor market cycles)
    - USOIL: Oil Prices (commodity price cycles)
    
    Target:
    - Annual Inflation Rate (target for business cycle forecasting)
    
    Note: Baxter-King filter will extract business cycle frequencies (6-32 quarters)
    while removing both low-frequency trends and high-frequency noise
    """
    print("\n📊 LOADING FED INDICATORS FOR BAXTER-KING FILTERING")
    print("-" * 55)
    
    try:
        data = pd.read_csv(file_path)
        print(f"✅ Data loaded: {data.shape[0]} observations, {data.shape[1]} features")
    except FileNotFoundError:
        print(f"❌ File not found: {file_path}")
        print("📁 Please check the file path and ensure the CSV file exists")
        return None
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        return None
    
    # Handle missing values
    missing_before = data.isnull().sum().sum()
    print(f"📋 Missing values before: {missing_before}")
    
    if missing_before > 0:
        data = data.interpolate(method='linear')
        data = data.fillna(method='ffill').fillna(method='bfill')
        
    missing_after = data.isnull().sum().sum()
    print(f"✅ Missing values after interpolation: {missing_after}")
    
    return data

# ================================================
# 2. SCIENTIFIC VARIABLE DEFINITION - SAME AS HP/KALMAN/IIR CODES
# ================================================

# LEADING FED INDICATORS (no inflation as predictor)
LEADING_INDICATORS = [
    'USM2',      # Money Supply M2 - Monetary policy stance
    'USPPIYY',   # Producer Price Index YoY - Cost pressures  
    'INDPRO',    # Industrial Production - Economic activity
    'UNRATE',    # Unemployment Rate - Labor market conditions
    'USOIL'      # Oil Prices - Supply shocks and commodity inflation
]

TARGET_VARIABLE = 'Annual Inflation Rate'

print(f"\n🎯 BAXTER-KING FILTER RESEARCH DESIGN:")
print(f"Leading Fed Indicators (Input Signals): {LEADING_INDICATORS}")
print(f"Target Variable (Output Signal): {TARGET_VARIABLE}")
print(f"\n💡 Research Question:")
print(f"Can Baxter-King bandpass filtering enhance ML/DL inflation forecasting")
print(f"by isolating business cycle frequencies from Fed economic indicators?")

# ================================================
# 3. BAXTER-KING FILTER CONFIGURATIONS
# ================================================

BK_FILTER_CONFIGS = {
    'low_frequency': 6,        # Minimum business cycle period (6 quarters = 1.5 years)
    'high_frequency': 32,      # Maximum business cycle period (32 quarters = 8 years)
    'leads_lags': 12,          # Number of leads and lags (K parameter)
    'filter_type': 'bandpass', # Bandpass filter (removes trends and noise)
    'symmetric': True,         # Symmetric weights (no phase distortion)
    'theory': 'Isolates business cycle frequencies while removing trends and noise'
}

print(f"\n🔧 BAXTER-KING FILTER SPECIFICATIONS:")
print(f"   Filter Type: {BK_FILTER_CONFIGS['filter_type']} (business cycle isolation)")
print(f"   Business Cycle Range: {BK_FILTER_CONFIGS['low_frequency']}-{BK_FILTER_CONFIGS['high_frequency']} quarters")
print(f"   Leads/Lags (K): {BK_FILTER_CONFIGS['leads_lags']} (symmetric weights)")
print(f"   Phase Distortion: None (symmetric filter)")
print(f"   Theory: {BK_FILTER_CONFIGS['theory']}")

# ================================================
# 4. MODEL CONFIGURATIONS (MAINTAINED FROM ORIGINAL)
# ================================================

MODEL_CONFIGS = {
    'LSTM': {
        'units': 50,
        'activation': 'tanh',
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'layers': 2,
        'architecture': 'Sequential: LSTM(50) → LSTM(50) → Dense(1)',
        'optimal_for': 'Long-term dependencies in business cycle patterns'
    },
    'GRU': {
        'units': 50,
        'activation': 'tanh', 
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'layers': 2,
        'architecture': 'Sequential: GRU(50) → GRU(50) → Dense(1)',
        'optimal_for': 'Efficient processing of cyclical patterns'
    },
    'CNN': {
        'filters': 64,
        'kernel_size': 2,
        'activation': 'relu',
        'dense_units': 50,
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'input_type': '1D convolution',
        'architecture': 'Conv1D(64) → Flatten → Dense(50) → Dense(1)',
        'optimal_for': 'Local pattern detection in business cycles'
    },
    'RNN': {
        'units': 50,
        'activation': 'tanh',
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'layers': 2,
        'architecture': 'Sequential: RNN(50) → RNN(50) → Dense(1)',
        'optimal_for': 'Basic sequential processing of cycle data'
    },
    'ANN': {
        'units': 50,
        'activation': 'relu',
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'layers': 2,
        'architecture': 'Dense(50) → Dense(50) → Dense(1)',
        'optimal_for': 'Non-linear relationships in cycle feature space'
    }
}

print(f"\n🤖 MODEL ARCHITECTURES FOR BAXTER-KING FILTERED FED DATA:")
for model_name, config in MODEL_CONFIGS.items():
    print(f"   {model_name}: {config['architecture']}")
    print(f"     └─ Optimal for: {config['optimal_for']}")

# ================================================
# 5. ENHANCED BAXTER-KING FILTER IMPLEMENTATION
# ================================================

def apply_baxter_king_filter(data, low_freq=6, high_freq=32, K=12):
    """
    Apply Baxter-King bandpass filter for business cycle analysis
    
    The Baxter-King filter is a bandpass filter designed specifically for 
    economic time series analysis. It isolates business cycle frequencies
    while removing both secular trends and high-frequency noise.
    
    Economic Theory:
    ----------------
    - Business cycles typically occur at frequencies of 6-32 quarters
    - Low frequencies (<6 quarters): High-frequency noise and seasonal effects
    - Target frequencies (6-32 quarters): Business cycle fluctuations  
    - High frequencies (>32 quarters): Long-term trends and structural changes
    
    Mathematical Foundation:
    ------------------------
    The filter uses symmetric weights to avoid phase distortion:
    
    w₀ = (ωₕ - ωₗ) / π                    (central weight)
    wₖ = [sin(kωₙ) - sin(kωₗ)] / (kπ)     (symmetric weights, k = 1,2,...,K)
    
    where ωₗ = 2π/high_freq and ωₕ = 2π/low_freq
    
    Parameters:
    -----------
    data : array-like
        Economic time series (Fed indicators)
    low_freq : int
        Minimum period of business cycle (quarters, default=6)
        Corresponds to 1.5 years (short business cycles)
    high_freq : int  
        Maximum period of business cycle (quarters, default=32)
        Corresponds to 8 years (long business cycles)
    K : int
        Number of leads and lags (default=12)
        Determines filter length = 2K+1
        
    Returns:
    --------
    np.array : Baxter-King filtered business cycle component
    
    Reference:
    ----------
    Baxter, M. and King, R.G. (1999). "Measuring Business Cycles: 
    Approximate Band-Pass Filters for Economic Time Series."
    Review of Economics and Statistics, 81(4), 575-593.
    
    Applications in Central Banking:
    --------------------------------
    - Federal Reserve: Business cycle dating and analysis
    - ECB: Euro area business cycle synchronization
    - Academic research: International business cycle correlations
    """
    try:
        print(f"      🔧 Applying Baxter-King filter (cycle: {low_freq}-{high_freq}Q, K={K})")
        
        # Convert periods to angular frequencies
        omega_low = 2 * np.pi / high_freq   # Low cutoff (long periods, trends)
        omega_high = 2 * np.pi / low_freq   # High cutoff (short periods, noise)
        
        # Validate frequency parameters
        if omega_high >= np.pi:
            print(f"      ⚠️ High frequency too high, adjusting low_freq to {low_freq+1}")
            omega_high = 2 * np.pi / (low_freq + 1)
        
        # Initialize weight vector
        weights = np.zeros(2*K + 1)
        
        # Central weight (k=0)
        weights[K] = (omega_high - omega_low) / np.pi
        
        # Symmetric weights (k = ±1, ±2, ..., ±K)
        for k in range(1, K+1):
            weight_k = (np.sin(k * omega_high) - np.sin(k * omega_low)) / (k * np.pi)
            weights[K + k] = weight_k  # Positive lags
            weights[K - k] = weight_k  # Negative lags (symmetry)
        
        # Normalize weights to ensure they sum to zero (removes DC component)
        # This is crucial for business cycle analysis
        weight_sum = np.sum(weights)
        if abs(weight_sum) > 1e-10:  # Adjust central weight if needed
            weights[K] -= weight_sum
        
        # Apply filter using convolution
        # Method 1: Direct convolution (handles edge effects)
        if len(data) > 2*K:
            # Pad data to handle edge effects
            padded_data = np.pad(data, K, mode='edge')
            filtered_data = np.convolve(padded_data, weights, mode='valid')
        else:
            print(f"      ⚠️ Data too short for K={K}, using smaller K")
            K_adjusted = min(K, len(data)//4)
            return apply_baxter_king_filter(data, low_freq, high_freq, K_adjusted)
        
        # Ensure output length matches input
        if len(filtered_data) != len(data):
            if len(filtered_data) > len(data):
                # Trim excess
                start_idx = (len(filtered_data) - len(data)) // 2
                filtered_data = filtered_data[start_idx:start_idx + len(data)]
            else:
                # Pad if needed (shouldn't happen with proper implementation)
                pad_needed = len(data) - len(filtered_data)
                filtered_data = np.pad(filtered_data, 
                                     (pad_needed//2, pad_needed - pad_needed//2), 
                                     mode='edge')
        
        # Diagnostic information
        cycle_variance = np.var(filtered_data)
        original_variance = np.var(data)
        variance_ratio = cycle_variance / original_variance
        
        print(f"      📊 Business cycle variance ratio: {variance_ratio:.3f}")
        print(f"      📊 Cycle component std: {np.std(filtered_data):.4f}")
        
        return filtered_data
        
    except Exception as e:
        print(f"      ⚠️ Baxter-King Filter error: {e}")
        print(f"      ↳ Returning original data (fallback)")
        return data

def analyze_business_cycle_properties(data, filtered_data, indicator_name):
    """
    Analyze business cycle properties of Baxter-King filtered data
    """
    print(f"      📈 Business Cycle Analysis for {indicator_name}:")
    
    # Calculate cycle statistics
    cycle_mean = np.mean(filtered_data)
    cycle_std = np.std(filtered_data)
    cycle_range = np.max(filtered_data) - np.min(filtered_data)
    
    # Peak and trough analysis
    from scipy.signal import find_peaks
    peaks, _ = find_peaks(filtered_data)
    troughs, _ = find_peaks(-filtered_data)
    
    avg_cycle_length = 0
    if len(peaks) > 1:
        avg_cycle_length = np.mean(np.diff(peaks))
    
    print(f"         • Cycle Mean: {cycle_mean:.4f}")
    print(f"         • Cycle Std: {cycle_std:.4f}")
    print(f"         • Cycle Range: {cycle_range:.4f}")
    print(f"         • Number of Peaks: {len(peaks)}")
    print(f"         • Number of Troughs: {len(troughs)}")
    print(f"         • Avg Cycle Length: {avg_cycle_length:.1f} periods")
    
    return {
        'cycle_mean': cycle_mean,
        'cycle_std': cycle_std,
        'cycle_range': cycle_range,
        'n_peaks': len(peaks),
        'n_troughs': len(troughs),
        'avg_cycle_length': avg_cycle_length
    }

# ================================================
# 6. PREPROCESSING FUNCTIONS (MAINTAINED)
# ================================================

def create_sequences(data, seq_length):
    """
    Create sequences for time series prediction from Baxter-King filtered Fed data
    
    Input structure after Baxter-King filtering:
    - Each sequence contains seq_length time steps of business cycle components
    - Features: BK-filtered leading indicators (USM2, USPPIYY, INDPRO, UNRATE, USOIL cycles)
    - Target: Annual Inflation Rate at time t+1
    
    Maintains temporal structure while focusing on business cycle frequencies
    """
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length, :-1])  # BK-filtered Fed cycle indicators
        y.append(data[i + seq_length, -1])     # Target: inflation at t+1
    return np.array(X), np.array(y)

# ================================================
# 7. MODEL BUILDERS (MAINTAINED WITH ENHANCEMENTS)
# ================================================

def build_lstm_model(input_shape):
    """Build LSTM model optimized for Baxter-King filtered business cycle data"""
    model = Sequential()
    model.add(LSTM(50, return_sequences=True, input_shape=input_shape, activation='tanh'))
    model.add(Dropout(0.2))  # Regularization for cycle patterns
    model.add(LSTM(50, activation='tanh'))
    model.add(Dropout(0.2))  # Prevent overfitting on cyclical patterns
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_gru_model(input_shape):
    """Build GRU model optimized for Baxter-King filtered cycle time series"""
    model = Sequential()
    model.add(GRU(50, return_sequences=True, input_shape=input_shape, activation='tanh'))
    model.add(Dropout(0.2))
    model.add(GRU(50, activation='tanh'))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_cnn_model(input_shape):
    """Build CNN model for 1D convolution on Baxter-King filtered sequences"""
    model = Sequential()
    model.add(Conv1D(64, kernel_size=2, activation='relu', input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(Flatten())
    model.add(Dense(50, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_rnn_model(input_shape):
    """Build RNN model for Baxter-King filtered business cycle series"""
    model = Sequential()
    model.add(SimpleRNN(50, return_sequences=True, input_shape=input_shape, activation='tanh'))
    model.add(Dropout(0.2))
    model.add(SimpleRNN(50, activation='tanh'))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_ann_model(input_shape):
    """Build ANN model for flattened Baxter-King filtered sequences"""
    model = Sequential()
    model.add(Dense(50, activation='relu', input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(Dense(50, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

# ================================================
# 8. COMPREHENSIVE METRICS
# ================================================

def calculate_comprehensive_metrics(y_true, y_pred):
    """Calculate comprehensive evaluation metrics for Baxter-King enhanced forecasting"""
    return {
        'mse': mean_squared_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae': mean_absolute_error(y_true, y_pred),
        'r2': r2_score(y_true, y_pred),
        'mape': mean_absolute_percentage_error(y_true, y_pred) * 100
    }

# ================================================
# 9. ML BASELINE MODELS
# ================================================

def train_ml_baselines(X_train_flat, X_test_flat, y_train, y_test):
    """Train ML baseline models on Baxter-King filtered Fed indicators"""
    print("   🌳 Training ML baselines on BK-filtered Fed cycle data...")
    ml_results = {}
    
    try:
        # Random Forest
        rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf_model.fit(X_train_flat, y_train)
        rf_pred = rf_model.predict(X_test_flat)
        ml_results['random_forest'] = calculate_comprehensive_metrics(y_test, rf_pred)
        
        # XGBoost
        xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        xgb_model.fit(X_train_flat, y_train)
        xgb_pred = xgb_model.predict(X_test_flat)
        ml_results['xgboost'] = calculate_comprehensive_metrics(y_test, xgb_pred)
        
        print(f"      ✅ Random Forest: MSE={ml_results['random_forest']['mse']:.6f}")
        print(f"      ✅ XGBoost: MSE={ml_results['xgboost']['mse']:.6f}")
        
    except Exception as e:
        print(f"      ⚠️ ML baseline error: {e}")
        
    return ml_results

# ================================================
# 10. MAIN ANALYSIS - BAXTER-KING FILTERING + FED INDICATORS
# ================================================

def run_baxter_king_fed_inflation_forecasting():
    """
    Run comprehensive inflation forecasting with Baxter-King filtering on Fed indicators
    """
    
    # Load data
    file_path = r"C:\Users\elect\OneDrive\Documentos\Doctorado\Articulo 2 peer review\Data\Inflation_Parameters.csv"
    print(f"📁 Loading Fed economic indicators from: {file_path}")
    
    merged_data = load_and_document_data(file_path)
    
    if merged_data is None:
        print("❌ Cannot proceed without valid Fed data")
        return []
    
    # Verify required columns exist
    required_columns = LEADING_INDICATORS + [TARGET_VARIABLE]
    missing_columns = [col for col in required_columns if col not in merged_data.columns]
    
    if missing_columns:
        print(f"❌ Missing required Fed indicators: {missing_columns}")
        print(f"📋 Available columns: {list(merged_data.columns)}")
        return []
    else:
        print(f"✅ All required Fed indicators found in dataset")
    
    results = []
    cycle_analysis_results = []
    seq_length = 50
    
    # Test both normalization methods
    scalers = {
        'MinMax': MinMaxScaler(),
        'Standard': StandardScaler()
    }
    
    print(f"\n🔬 BAXTER-KING FED INFLATION FORECASTING EXPERIMENT")
    print(f"Fed Leading indicators: {LEADING_INDICATORS}")
    print(f"Target: {TARGET_VARIABLE}")
    print(f"Business cycle range: {BK_FILTER_CONFIGS['low_frequency']}-{BK_FILTER_CONFIGS['high_frequency']} quarters")
    print(f"Sequence length: {seq_length}")
    print(f"Normalization methods: {list(scalers.keys())}")
    print("-" * 75)
    
    # Calculate total experiments
    total_combinations = sum(1 for r in range(1, len(LEADING_INDICATORS) + 1) 
                           for _ in combinations(LEADING_INDICATORS, r))
    total_experiments = total_combinations * len(scalers)
    current_experiment = 0
    
    # Iterate through all combinations of Fed leading indicators
    for r in range(1, len(LEADING_INDICATORS) + 1):
        for combo in combinations(LEADING_INDICATORS, r):
            for scaler_name, scaler in scalers.items():
                current_experiment += 1
                
                print(f"\n📊 Experiment {current_experiment}/{total_experiments}")
                print(f"    Fed Indicators: {combo}")
                print(f"    Normalization: {scaler_name}")
                
                try:
                    # Prepare dataset with ONLY Fed leading indicators + target
                    selected_features = list(combo) + [TARGET_VARIABLE]
                    selected_data = merged_data[selected_features].copy()
                    
                    print(f"   📊 Selected Fed data shape: {selected_data.shape}")
                    
                    # Apply Baxter-King Filter to each Fed indicator
                    print("   🔧 Applying Baxter-King Filter to Fed economic indicators...")
                    cycle_stats = {}
                    
                    for column in selected_features:
                        filtered_data = apply_baxter_king_filter(
                            selected_data[column].values,
                            low_freq=BK_FILTER_CONFIGS['low_frequency'],
                            high_freq=BK_FILTER_CONFIGS['high_frequency'],
                            K=BK_FILTER_CONFIGS['leads_lags']
                        )
                        
                        # Business cycle analysis
                        if column in LEADING_INDICATORS:  # Don't analyze target variable
                            cycle_stats[column] = analyze_business_cycle_properties(
                                selected_data[column].values, filtered_data, column
                            )
                        
                        # Ensure consistent length after filtering
                        if len(filtered_data) != len(selected_data[column]):
                            filtered_data = np.resize(filtered_data, len(selected_data[column]))
                        selected_data[column] = filtered_data
                    
                    # Store cycle analysis results
                    cycle_analysis_results.append({
                        'experiment': current_experiment,
                        'indicators': combo,
                        'scaler': scaler_name,
                        'cycle_stats': cycle_stats
                    })
                    
                    # Normalize Baxter-King filtered Fed data
                    scaled_data = scaler.fit_transform(selected_data)
                    
                    # Create sequences for time series modeling
                    X, y = create_sequences(scaled_data, seq_length)
                    
                    if len(X) < 20:  # Minimum samples for reliable training
                        print("   ⚠️ Insufficient data for reliable training")
                        continue
                    
                    # Temporal train-test split (crucial for time series)
                    split_idx = int(len(X) * 0.8)
                    X_train, X_test = X[:split_idx], X[split_idx:]
                    y_train, y_test = y[:split_idx], y[split_idx:]
                    
                    print(f"   📊 Data split: Train={len(X_train)}, Test={len(X_test)}")
                    
                    # Prepare input shapes
                    input_shape_seq = (seq_length, X_train.shape[2])
                    X_train_flat = X_train.reshape(X_train.shape[0], -1)
                    X_test_flat = X_test.reshape(X_test.shape[0], -1)
                    input_shape_ann = (X_train_flat.shape[1],)
                    
                    # Early stopping callback
                    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
                    
                    # Train all models
                    models_results = []
                    
                    # Neural Network Models
                    neural_models = [
                        ('LSTM', build_lstm_model, X_train, X_test, input_shape_seq),
                        ('GRU', build_gru_model, X_train, X_test, input_shape_seq),
                        ('CNN', build_cnn_model, X_train, X_test, input_shape_seq),
                        ('RNN', build_rnn_model, X_train, X_test, input_shape_seq),
                        ('ANN', build_ann_model, X_train_flat, X_test_flat, input_shape_ann)
                    ]
                    
                    for model_name, model_builder, X_tr, X_te, input_shape in neural_models:
                        print(f"   🤖 Training {model_name} on BK-filtered Fed cycle data...")
                        
                        try:
                            model = model_builder(input_shape)
                            history = model.fit(X_tr, y_train, epochs=100, batch_size=32, 
                                              validation_split=0.2, verbose=0, callbacks=[early_stopping])
                            
                            y_pred = model.predict(X_te, verbose=0).flatten()
                            metrics = calculate_comprehensive_metrics(y_test, y_pred)
                            models_results.append((model_name, metrics))
                            
                            print(f"      ✅ {model_name}: MSE={metrics['mse']:.6f}, R²={metrics['r2']:.4f}")
                            
                        except Exception as e:
                            print(f"      ❌ {model_name} failed: {e}")
                    
                    # Train ML baseline models
                    ml_results = train_ml_baselines(X_train_flat, X_test_flat, y_train, y_test)
                    for ml_model, ml_metrics in ml_results.items():
                        models_results.append((ml_model.upper(), ml_metrics))
                    
                    # Store all results
                    for model_name, metrics in models_results:
                        result = {
                            'leading_indicators': combo,
                            'n_indicators': len(combo),
                            'scaler': scaler_name,
                            'model': model_name,
                            'seq_length': seq_length if model_name not in ['RANDOM_FOREST', 'XGBOOST'] else 'N/A',
                            'filter_applied': 'Baxter_King',
                            'business_cycle_range': f"{BK_FILTER_CONFIGS['low_frequency']}-{BK_FILTER_CONFIGS['high_frequency']}Q",
                            'leads_lags': BK_FILTER_CONFIGS['leads_lags'],
                            **metrics
                        }
                        results.append(result)
                
                except Exception as e:
                    print(f"   ❌ Experiment failed: {e}")
                    continue
    
    return results, cycle_analysis_results

# ================================================
# 11. EXECUTE ANALYSIS AND CREATE VISUALIZATIONS
# ================================================

if __name__ == "__main__":
    print("🚀 STARTING BAXTER-KING FED INFLATION FORECASTING ANALYSIS")
    
    try:
        # Run the analysis
        results, cycle_analysis = run_baxter_king_fed_inflation_forecasting()
        
        if not results:
            print("❌ No results generated!")
        else:
            # Convert to DataFrame
            results_df = pd.DataFrame(results)
            
            # Save results
            output_path = r'C:\Users\elect\OneDrive\Documentos\Doctorado\Articulo 2 peer review\results\baxter_king_fed_inflation_results.csv'
            results_df.to_csv(output_path, index=False)
            print(f"\n💾 Results saved to: {output_path}")
            
            # Save cycle analysis
            import json
            cycle_path = r'C:\Users\elect\OneDrive\Documentos\Doctorado\Articulo 2 peer review\results\baxter_king_cycle_analysis.json'
            with open(cycle_path, 'w') as f:
                json.dump(cycle_analysis, f, indent=2, default=str)
            print(f"💾 Cycle analysis saved to: {cycle_path}")
            
            # ================================================
            # 12. ENHANCED BAXTER-KING VISUALIZATION
            # ================================================
            
            print(f"\n📊 CREATING BAXTER-KING FED ANALYSIS VISUALIZATIONS")
            print("-" * 55)
            
            # Create comprehensive visualization
            fig, axes = plt.subplots(2, 3, figsize=(20, 12))
            fig.suptitle('Baxter-King Enhanced Inflation Forecasting\n(Fed Leading Indicators - Business Cycle Analysis)', 
                         fontsize=16, fontweight='bold')
            
            # 1. Model Performance with Baxter-King Filtering (MSE)
            ax1 = axes[0, 0]
            model_mse = results_df.groupby('model')['mse'].mean().sort_values()
            bars1 = ax1.bar(range(len(model_mse)), model_mse.values, 
                            color=plt.cm.Set3(np.linspace(0, 1, len(model_mse))))
            ax1.set_xticks(range(len(model_mse)))
            ax1.set_xticklabels(model_mse.index, rotation=45, ha='right')
            ax1.set_ylabel('Mean Squared Error')
            ax1.set_title('Baxter-King Business Cycle Forecasting')
            ax1.grid(True, alpha=0.3)
            
            # Add value labels
            for bar, value in zip(bars1, model_mse.values):
                ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                        f'{value:.4f}', ha='center', va='bottom', fontsize=9)
            
            # 2. R² with Baxter-King Filtering
            ax2 = axes[0, 1]
            model_r2 = results_df.groupby('model')['r2'].mean().sort_values(ascending=False)
            bars2 = ax2.bar(range(len(model_r2)), model_r2.values,
                            color=plt.cm.Set2(np.linspace(0, 1, len(model_r2))))
            ax2.set_xticks(range(len(model_r2)))
            ax2.set_xticklabels(model_r2.index, rotation=45, ha='right')
            ax2.set_ylabel('R² Score')
            ax2.set_title('Explained Variance (BK Cycle-Filtered)')
            ax2.grid(True, alpha=0.3)
            
            # 3. Fed Indicators Impact with Baxter-King
            ax3 = axes[0, 2]
            indicator_impact = results_df.groupby('n_indicators')['r2'].agg(['mean', 'std']).reset_index()
            ax3.errorbar(indicator_impact['n_indicators'], indicator_impact['mean'], 
                        yerr=indicator_impact['std'], marker='o', capsize=5, linewidth=2)
            ax3.set_xlabel('Number of BK-Filtered Fed Cycle Indicators')
            ax3.set_ylabel('R² Score (Mean ± Std)')
            ax3.set_title('Business Cycle Complexity vs Performance')
            ax3.grid(True, alpha=0.3)
            
            # 4. Scaler Impact on Baxter-King Fed Data
            ax4 = axes[1, 0]
            scaler_perf = results_df.groupby(['model', 'scaler'])['r2'].mean().unstack()
            scaler_perf.plot(kind='bar', ax=ax4, width=0.8)
            ax4.set_xlabel('Model Type')
            ax4.set_ylabel('R² Score')
            ax4.set_title('Normalization Impact (Post-BK)')
            ax4.legend(title='Scaler')
            ax4.grid(True, alpha=0.3)
            plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')
            
            # 5. Top Baxter-King Fed Configurations
            ax5 = axes[1, 1]
            top_10 = results_df.nsmallest(10, 'mse')
            config_names = [f"{row['model']}\n{'+'.join(row['leading_indicators'][:2])}" 
                           for _, row in top_10.iterrows()]
            bars5 = ax5.barh(range(len(top_10)), top_10['mse'])
            ax5.set_yticks(range(len(top_10)))
            ax5.set_yticklabels(config_names, fontsize=8)
            ax5.set_xlabel('MSE')
            ax5.set_title('Top 10 BK Business Cycle Configurations')
            ax5.grid(True, alpha=0.3)
            
            # 6. MAPE Distribution for Baxter-King Fed Models
            ax6 = axes[1, 2]
            results_df.boxplot(column='mape', by='model', ax=ax6)
            ax6.set_xlabel('Model Type')
            ax6.set_ylabel('MAPE (%)')
            ax6.set_title('BK Business Cycle Error Distribution')
            ax6.grid(True, alpha=0.3)
            plt.setp(ax6.xaxis.get_majorticklabels(), rotation=45, ha='right')
            
            plt.tight_layout()
            plt.savefig('baxter_king_fed_inflation_analysis.png', dpi=300, bbox_inches='tight')
            plt.show()
            
            # ================================================
            # 13. BAXTER-KING FED SCIENTIFIC REPORT
            # ================================================
            
            print(f"\n📋 BAXTER-KING FED SCIENTIFIC ANALYSIS REPORT")
            print("="*75)
            
            print(f"\n🎯 RESEARCH OBJECTIVE:")
            print(f"Evaluate Baxter-King bandpass filtering for business cycle analysis")
            print(f"in inflation forecasting using Fed leading indicators (no data leakage)")
            
            print(f"\n🔧 BAXTER-KING FILTER METHODOLOGY:")
            print(f"   • Filter Type: Bandpass filter for business cycle isolation")
            print(f"   • Frequency Range: {BK_FILTER_CONFIGS['low_frequency']}-{BK_FILTER_CONFIGS['high_frequency']} quarters")
            print(f"   • Leads/Lags: {BK_FILTER_CONFIGS['leads_lags']} (symmetric weights)")
            print(f"   • Theory: Isolates business cycle frequencies")
            print(f"   • Purpose: Remove trends and noise, preserve cyclical patterns")
            
            print(f"\n📊 EXPERIMENTAL SETUP:")
            print(f"   Total experiments: {len(results_df)}")
            print(f"   Models tested: {results_df['model'].nunique()}")
            print(f"   BK-filtered Fed combinations: {results_df['leading_indicators'].nunique()}")
            print(f"   Best MSE achieved: {results_df['mse'].min():.6f}")
            print(f"   Best R² achieved: {results_df['r2'].max():.6f}")
            
            print(f"\n🏆 TOP 5 BAXTER-KING FED CONFIGURATIONS:")
            top_5 = results_df.nsmallest(5, 'mse')
            for i, (_, row) in enumerate(top_5.iterrows(), 1):
                indicators_str = '+'.join(row['leading_indicators'])
                print(f"{i}. {row['model']} | BK Fed Cycles: {indicators_str}")
                print(f"   Scaler: {row['scaler']} | MSE: {row['mse']:.6f} | R²: {row['r2']:.4f}")
                print(f"   MAE: {row['mae']:.4f} | MAPE: {row['mape']:.2f}%")
                print("-" * 75)
            
            print(f"\n🤖 BAXTER-KING ENHANCED FED MODEL RANKING:")
            model_ranking = results_df.groupby('model').agg({
                'mse': ['mean', 'std'],
                'r2': ['mean', 'std'],
                'mape': ['mean', 'std']
            }).round(6)
            print(model_ranking)
            
            print(f"\n📈 BAXTER-KING FED KEY FINDINGS:")
            best_config = results_df.loc[results_df['mse'].idxmin()]
            print(f"   • Best BK-enhanced model: {best_config['model']}")
            print(f"   • Optimal Fed cycle indicators: {'+'.join(best_config['leading_indicators'])}")
            print(f"   • Best normalization: {best_config['scaler']}")
            
            print(f"\n💡 BAXTER-KING ADVANTAGES FOR FED DATA:")
            print(f"   • Isolates business cycle frequencies from Fed indicators")
            print(f"   • Removes both trends and high-frequency noise")
            print(f"   • Symmetric weights ensure no phase distortion")
            print(f"   • Standard methodology in business cycle research")
            print(f"   • Directly applicable to NBER business cycle dating")
            
            print(f"\n🔬 SCIENTIFIC CONTRIBUTION:")
            print(f"   • Novel BK application to Fed inflation forecasting")
            print(f"   • Business cycle frequency analysis for inflation")
            print(f"   • Comparison with HP and Kalman filtering approaches")
            print(f"   • Cyclical pattern extraction for Fed indicators")
            print(f"   • Methodology for central bank cycle-based forecasting")

    except Exception as e:
        print(f"❌ Error during Baxter-King analysis: {e}")
        import traceback
        traceback.print_exc()

print(f"\n🎯 BAXTER-KING FED INFLATION FORECASTING COMPLETED!")
print("📊 Business cycle frequency analysis for optimal Fed indicator filtering")
print("🔄 Results enable comprehensive comparison with HP and Kalman approaches")
print("🏦 Cyclical methodology applicable to Federal Reserve business cycle analysis")

🚀 ROBUST INFLATION FORECASTING WITH BAXTER-KING FILTERING
✅ NO inflation as predictor (avoiding data leakage)
✅ Using Baxter-King bandpass filter for business cycle analysis
✅ Multiple evaluation metrics (MSE, MAE, R², MAPE)
✅ Scientific methodology for policy applications
✅ Business cycle frequency isolation from Fed indicators

🎯 BAXTER-KING FILTER RESEARCH DESIGN:
Leading Fed Indicators (Input Signals): ['USM2', 'USPPIYY', 'INDPRO', 'UNRATE', 'USOIL']
Target Variable (Output Signal): Annual Inflation Rate

💡 Research Question:
Can Baxter-King bandpass filtering enhance ML/DL inflation forecasting
by isolating business cycle frequencies from Fed economic indicators?

🔧 BAXTER-KING FILTER SPECIFICATIONS:
   Filter Type: bandpass (business cycle isolation)
   Business Cycle Range: 6-32 quarters
   Leads/Lags (K): 12 (symmetric weights)
   Phase Distortion: None (symmetric filter)
   Theory: Isolates business cycle frequencies while removing trends and noise

🤖 MODEL ARCHITECTURES FOR B

KeyboardInterrupt: 